# 01 — Build Manifest
# Giai đoạn 1 — Mục 1.1 — Đọc manifest đã lọc (hoặc xây dựng lại)
 **Đầu ra**: `outputs/tables/manifest_filtered.csv`

In [1]:
from pathlib import Path
import pandas as pd
from common import pipeline, io_utils

In [2]:
USE_SYNTHETIC_DATA = False
REAL_DATA_ROOT = Path("../../data/raw")
SYNTHETIC_DATA_ROOT = Path("./_data/synthetic_cwru")
OUTPUT_DIR = Path("./outputs")
FORCE_REBUILD_MANIFEST = False  # Đặt True chỉ khi muốn quét lại data/raw từ đầu

TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

- NẾU MANIFEST ĐÃ TỒN TẠI

In [3]:
manifest_path = TABLES_DIR / "manifest_filtered.csv"

# Cột BẮT BUỘC phải có trong manifest. Thiếu bất kỳ cột nào nghĩa là file
# cache này được sinh bởi bản io_utils CŨ (thời còn đoán fs = 12000Hz cho mọi
# file, kể cả Normal baseline 48kHz, và còn dùng khóa 'sensor_position'), nên
# phải quét lại data/raw thay vì dùng tiếp. Cache im lặng là cách nhanh nhất
# để mọi thứ đã sửa ở common/ không có tác dụng gì.
REQUIRED_MANIFEST_COLS = ("resolved_sample_rate_hz", "sensor_location")

use_cache = manifest_path.exists() and not FORCE_REBUILD_MANIFEST
if use_cache:
    manifest = pd.read_csv(manifest_path)
    missing = [c for c in REQUIRED_MANIFEST_COLS if c not in manifest.columns]
    if missing:
        print(f"[CACHE CŨ] {manifest_path.name} thiếu cột {missing} - quét lại data/raw.")
        use_cache = False

if use_cache:
    print(f"Đã đọc manifest từ: {manifest_path}")
else:
    # Xây dựng manifest mới
    manifest_full = pipeline.get_manifest(
        use_synthetic=USE_SYNTHETIC_DATA,
        real_data_root=REAL_DATA_ROOT,
        synthetic_data_root=SYNTHETIC_DATA_ROOT,
        output_dir=OUTPUT_DIR,
        force_rebuild=True,
    )
    manifest = io_utils.apply_scope_filter(manifest_full)
    manifest.to_csv(manifest_path, index=False)
    print(f"Đã xây dựng và lưu manifest mới: {manifest_path}")

print(f"Số file trong manifest: {len(manifest)}")
manifest.head()

Đã đọc manifest từ: outputs\tables\manifest_filtered.csv
Số file trong manifest: 40


,file_path,load_hp,label,fault_diameter_mils,or_position,source_category,sensor_location,declared_sample_rate_khz,n_samples_DE,n_samples_FE,n_samples_BA,rpm_from_file,read_error,warnings,has_warning,resolved_sample_rate_hz
0,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,0,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,122571,122571.0,122571.0,1796.0,NaN,NaN,False,12000.0
1,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,1,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121410,121410.0,121410.0,1772.0,NaN,NaN,False,12000.0
2,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,2,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1748.0,NaN,NaN,False,12000.0
3,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,3,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1722.0,NaN,NaN,False,12000.0
4,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,0,B,14.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121846,121846.0,121846.0,1796.0,NaN,NaN,False,12000.0


In [4]:
# --- Kiểm tra toàn vẹn manifest trước khi dùng cho các bước sau ---
assert manifest["file_path"].is_unique, "Có file_path trùng lặp trong manifest!"

required_cols = ["label", "load_hp", "rpm_from_file", "resolved_sample_rate_hz"]
na_counts = manifest[required_cols].isna().sum()
if na_counts.any():
    print("[CẢNH BÁO] Có giá trị thiếu ở cột bắt buộc:")
    print(na_counts[na_counts > 0])

# Phân bố lớp — kiểm tra cân bằng cho LOLO (RQ1) và phân loại 10 lớp (RQ2)
print("\nPhân bố label x load_hp:")
print(pd.crosstab(manifest["label"], manifest["load_hp"]))

# Các file còn cảnh báo NHẸ (không bị loại nhưng cần bạn tự xem xét)
still_warned = manifest[manifest["has_warning"]]
if len(still_warned):
    print(f"\n[XEM XÉT] {len(still_warned)} file còn cảnh báo chưa xử lý:")
    print(still_warned[["file_path", "warnings"]].to_string(index=False))
else:
    print("\nKhông còn file nào có cảnh báo trong tập đã lọc.")

[CẢNH BÁO] Có giá trị thiếu ở cột bắt buộc:
rpm_from_file    2
dtype: int64

Phân bố label x load_hp:
load_hp  0  1  2  3
label              
B        3  3  3  3
IR       3  3  3  3
Normal   1  1  1  1
OR       3  3  3  3

[XEM XÉT] 4 file còn cảnh báo chưa xử lý:
                             file_path                                                                                                                                                                              warnings
..\..\data\raw\Normal\100_Normal_3.mat NGHI_NGO_SAMPLING_RATE: n_samples=485643 (12kHz->40.5s, 24kHz->20.2s, 48kHz->10.1s). Rate hợp lý nhất: 48kHz — KHÁC rate mà file tự nhận (12kHz, qua thư mục nguồn/phạm vi mục tiêu).
 ..\..\data\raw\Normal\97_Normal_0.mat  NGHI_NGO_SAMPLING_RATE: n_samples=243938 (12kHz->20.3s, 24kHz->10.2s, 48kHz->5.1s). Rate hợp lý nhất: 24kHz — KHÁC rate mà file tự nhận (12kHz, qua thư mục nguồn/phạm vi mục tiêu).
 ..\..\data\raw\Normal\98_Normal_1.mat NGHI_NGO_SAMPLING_RATE: n_samples